# Data Clean Up - india_post_pincode_directory

**Source:** `virtue_foundation_dataset.bronze.india_post_pincode_directory`

**Target:** `virtue_foundation_dataset.silver.india_post_pincode_directory_cleaned` (overwrite)

## Steps
1. Load bronze table
2. Standardize all string fields to lower case
3. Naive deduplication - drop fully-duplicate rows, print count removed
4. Clean lat/long strings (trim whitespace, handle 'NA'), cast to DOUBLE
5. Backfill missing lat/long via internal join on (pincode + district + state) where other rows for the same combination have coordinates
6. Log remaining unresolved unique (officename, pincode, district, statename) combinations still missing lat/long
7. Write cleaned silver table (overwrite)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

BRONZE_TABLE = "virtue_foundation_dataset.bronze.india_post_pincode_directory"
SILVER_TABLE = "virtue_foundation_dataset.silver.india_post_pincode_directory_cleaned"

df_bronze = spark.table(BRONZE_TABLE)

print(f"Bronze row count: {df_bronze.count()}")
df_bronze.printSchema()

## Step 1: Standardize string fields to lower case

Applies `lower(trim(...))` to all string-typed columns. Numeric/coordinate columns (`pincode`, `latitude`, `longitude`) are handled separately below since they arrive as strings but represent numbers.

In [0]:
string_cols = [f.name for f in df_bronze.schema.fields if f.dataType.typeName() == "string"]
print(f"String columns to lower-case: {string_cols}")

df_lower = df_bronze
for col_name in string_cols:
    df_lower = df_lower.withColumn(col_name, F.lower(F.trim(F.col(col_name))))

df_lower.show(3, truncate=False)

## Step 2: Naive deduplication

Drop rows where *every* column is an exact duplicate of another row. Genie's profiling found 4 such exact duplicates (same PIN, office name, district, state, coordinates).

In [0]:
count_before_dedup = df_lower.count()

df_deduped = df_lower.dropDuplicates()

count_after_dedup = df_deduped.count()
removed_dedup = count_before_dedup - count_after_dedup

print(f"Rows before deduplication: {count_before_dedup}")
print(f"Rows after deduplication:  {count_after_dedup}")
print(f"Exact duplicate rows removed: {removed_dedup}")

## Step 3: Standardize coordinates - clean and cast to DOUBLE

- Trim whitespace from `latitude` / `longitude` strings
- Treat literal `"NA"` (any casing, after trim/lower) and empty strings as NULL
- Cast to DOUBLE

In [0]:
def clean_coord(col_name):
    raw = F.trim(F.col(col_name))
    raw = F.when((F.lower(raw) == "na") | (raw == ""), F.lit(None)).otherwise(raw)

    no_dir = F.regexp_replace(raw, r"\s*[NSEWnsew]\s*$", "")
    direction = F.upper(F.regexp_extract(raw, r"([NSEWnsew])\s*$", 1))

    # Remove ALL remaining internal whitespace (handles stray-space typos like "28.4 32628" -> "28.432628")
    no_dir = F.regexp_replace(no_dir, r"\s+", "")

    decimal_deg = F.regexp_extract(no_dir, r"^(-?\d+(?:\.\d+)?)\s*°?\s*$", 1)

    dms_match = F.regexp_extract(no_dir, r"^(-?\d+)\s*°\s*(\d+)\s*'\s*(\d+(?:\.\d+)?)\"?\s*$", 0)
    deg_part = F.regexp_extract(no_dir, r"^(-?\d+)\s*°", 1).cast(DoubleType())
    min_part = F.regexp_extract(no_dir, r"°\s*(\d+)\s*'", 1).cast(DoubleType())
    sec_part = F.regexp_extract(no_dir, r"'\s*(\d+(?:\.\d+)?)\"?\s*$", 1).cast(DoubleType())

    dms_value = deg_part + (min_part / 60.0) + (sec_part / 3600.0)

    parsed = (
        F.when(dms_match != "", dms_value)
         .when(decimal_deg != "", decimal_deg.cast(DoubleType()))
         .otherwise(no_dir)
    )

    parsed = F.when(direction.isin("S", "W"), -F.abs(parsed)).otherwise(parsed)

    cleaned_value = parsed.try_cast(DoubleType())

    return F.round(cleaned_value, 7)

df_coords = df_deduped \
    .withColumn("latitude", clean_coord("latitude")) \
    .withColumn("longitude", clean_coord("longitude"))

INDIA_LAT_RANGE = (6.0, 37.6)
INDIA_LON_RANGE = (68.0, 97.5)

lat_in_range = F.col("latitude").between(*INDIA_LAT_RANGE)
lon_in_range = F.col("longitude").between(*INDIA_LON_RANGE)

has_coords = F.col("latitude").isNotNull() & F.col("longitude").isNotNull()
in_range = lat_in_range & lon_in_range

# Rows with both coords present but out of India's bounding box (includes swapped lat/long) get dropped
out_of_range_coords = has_coords & ~in_range

n_out_of_range_coords = df_coords.filter(out_of_range_coords).count()
print(f"Rows with out-of-range (incl. swapped) lat/long dropped: {n_out_of_range_coords}")

df_coords = df_coords.filter(~out_of_range_coords)

missing_before = df_coords.filter(F.col("latitude").isNull() | F.col("longitude").isNull()).count()
total_rows = df_coords.count()

print(f"Total rows: {total_rows}")
print(f"Rows missing lat/long after cleaning: {missing_before}")
print(f"Percent missing: {missing_before / total_rows * 100:.2f}%")

## Step 4: Backfill missing lat/long via internal join

For rows still missing `latitude`/`longitude`, look for other rows in the **same file** sharing the same `(pincode, district, statename)` combination that *do* have coordinates. If found, use the average (centroid) of those known coordinates for that combination as the backfilled value.

We also add a `coordinate_source` column to track provenance:
- `original` - coordinate was present in bronze
- `backfilled_pincode_district_state` - coordinate was filled from a matching (pincode, district, statename) group
- `missing` - still unresolved after this step

In [0]:
# Tag rows with their original coordinate availability before backfill
df_tagged = df_coords.withColumn(
    "coordinate_source",
    F.when(F.col("latitude").isNotNull() & F.col("longitude").isNotNull(), F.lit("original"))
     .otherwise(F.lit("missing"))
)

join_keys = ["officename", "pincode", "district", "statename"]

# Build lookup of average coordinates per (pincode, district, statename) from rows that HAVE coordinates
coord_lookup = (
    df_tagged
    .filter(F.col("coordinate_source") == "original")
    .groupBy(*join_keys)
    .agg(
        F.avg("latitude").alias("lookup_latitude"),
        F.avg("longitude").alias("lookup_longitude")
    )
)

print(f"Distinct (officename, pincode, district, statename) groups with at least one known coordinate: {coord_lookup.count()}")

# Join missing rows to the lookup
df_backfilled = df_tagged.alias("main").join(
    coord_lookup.alias("lookup"),
    on=join_keys,
    how="left"
)

df_backfilled = df_backfilled.withColumn(
    "latitude",
    F.when(
        (F.col("main.coordinate_source") == "missing") & F.col("lookup_latitude").isNotNull(),
        F.col("lookup_latitude")
    ).otherwise(F.col("main.latitude"))
).withColumn(
    "longitude",
    F.when(
        (F.col("main.coordinate_source") == "missing") & F.col("lookup_longitude").isNotNull(),
        F.col("lookup_longitude")
    ).otherwise(F.col("main.longitude"))
).withColumn(
    "coordinate_source",
    F.when(
        (F.col("main.coordinate_source") == "missing") & F.col("lookup_latitude").isNotNull(),
        F.lit("backfilled_pincode_district_state")
    ).otherwise(F.col("main.coordinate_source"))
).drop("lookup_latitude", "lookup_longitude")

In [0]:
# Metrics on the backfill step
source_counts = df_backfilled.groupBy("coordinate_source").count().collect()
source_counts_dict = {row["coordinate_source"]: row["count"] for row in source_counts}

n_original = source_counts_dict.get("original", 0)
n_backfilled = source_counts_dict.get("backfilled_pincode_district_state", 0)
n_still_missing = source_counts_dict.get("missing", 0)

print(f"Total rows: {total_rows}")
print(f"Rows with original coordinates: {n_original}")
print(f"Rows backfilled via (officename, pincode, district, statename) match: {n_backfilled}")
print(f"Rows still missing lat/long: {n_still_missing}")
if missing_before > 0:
    print(f"Percent of originally-missing rows resolved by backfill: {n_backfilled / missing_before * 100:.2f}%")

## Step 5: Log unique combinations still missing lat/long

Print the number of unique `(officename, pincode, district, statename)` combinations that remain without coordinates after the backfill step.

In [0]:
unresolved = df_backfilled.filter(F.col("coordinate_source") == "missing")

unresolved_unique_combos = unresolved.select(
    "officename", "pincode", "district", "statename"
).distinct()

unresolved_unique_count = unresolved_unique_combos.count()
unresolved_row_count = unresolved.count()

print(f"Unique (officename, pincode, district, statename) combinations still missing lat/long: {unresolved_unique_count}")
print(f"Total rows still missing lat/long: {unresolved_row_count}")

unresolved_unique_combos.show(20, truncate=False)

## Step 5b: Drop rows with coordinates outside India's bounding box

After swap-correction, any row whose `latitude`/`longitude` still fall outside
India's approximate bounding box (lat 6.0-37.6, lon 68.0-97.5) is considered
unresolvable bad geodata and is dropped. Rows with NULL coordinates are kept
(they're tracked via `coordinate_source == "missing"`).

In [0]:
in_india_bbox = (
    F.col("latitude").isNull()
    | F.col("longitude").isNull()
    | (F.col("latitude").between(*INDIA_LAT_RANGE) & F.col("longitude").between(*INDIA_LON_RANGE))
)

count_before_bbox_filter = df_backfilled.count()

out_of_range = df_backfilled.filter(~in_india_bbox)
n_out_of_range = out_of_range.count()

print(f"Rows before bounding-box filter (post-backfill): {count_before_bbox_filter}")
print(f"Rows outside India bounding box from backfilled centroids (dropped): {n_out_of_range}")

In [0]:
from pyspark.sql.window import Window

KEY_COLS = ["officename", "pincode", "district", "statename", "circlename", "regionname", "divisionname"]

count_before_key_dedup = df_backfilled.count()

w = Window.partitionBy(*KEY_COLS).orderBy(
    F.col("latitude").isNull().asc(),
    F.col("longitude").isNull().asc()
)

df_key_deduped = (
    df_backfilled
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

count_after_key_dedup = df_key_deduped.count()
removed_key_dedup = count_before_key_dedup - count_after_key_dedup

print(f"Rows before key-column dedup: {count_before_key_dedup}")
print(f"Rows after key-column dedup:  {count_after_key_dedup}")
print(f"Duplicate rows removed (same officename/pincode/district/statename/circlename/regionname/divisionname): {removed_key_dedup}")


## Step 5c: Deduplicate on key business columns

For rows sharing the same `(officename, pincode, district, statename, circlename, regionname, divisionname)`,
keep only one record. If multiple coordinate variants exist within an in-range group,
arbitrarily keep the first one.

In [0]:
from pyspark.sql.window import Window

KEY_COLS = ["officename", "pincode", "district", "statename", "circlename", "regionname", "divisionname"]

count_before_key_dedup = df_backfilled.count()

w = Window.partitionBy(*KEY_COLS).orderBy(
    F.col("latitude").isNull().asc(),
    F.col("longitude").isNull().asc()
)

df_backfilled = (
    df_backfilled
    .withColumn("_rn", F.row_number().over(w))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

count_after_key_dedup = df_backfilled.count()
removed_key_dedup = count_before_key_dedup - count_after_key_dedup

print(f"Rows before key-column dedup: {count_before_key_dedup}")
print(f"Rows after key-column dedup:  {count_after_key_dedup}")
print(f"Duplicate rows removed (same officename/pincode/district/statename/circlename/regionname/divisionname): {removed_key_dedup}")

## Step 6: Write the cleaned silver table (overwrite)

In [0]:
df_final = df_backfilled.drop("coordinate_source")

print(f"Final row count: {df_final.count()}")
df_final.printSchema()

(
    df_final.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"Wrote cleaned table to {SILVER_TABLE}")

## Summary of cleanup metrics

In [0]:
print("=== Cleanup Summary ===")
print(f"Bronze row count:                 {count_before_dedup}")
print(f"Exact duplicate rows removed:     {removed_dedup}")
print(f"Rows after deduplication:         {count_after_dedup}")
print(f"Rows missing lat/long (raw):      {missing_before} ({missing_before / total_rows * 100:.2f}%)")
print(f"Rows backfilled (officename+pincode+district+state match): {n_backfilled}")
print(f"Rows still missing lat/long after backfill: {n_still_missing}")
print(f"Unique (officename, pincode, district, statename) combos still missing lat/long: {unresolved_unique_count}")
print(f"Rows dropped for out-of-range/swapped lat-long (pre-backfill): {n_out_of_range_coords}")
print(f"Rows dropped for out-of-range backfilled centroids (post-backfill): {n_out_of_range}")
print(f"Final silver table: {SILVER_TABLE}")